# TaskRouter Experiments

**Total cost: ~$1-2** using cheap models (Gemini Flash, GPT-4o-mini, Claude Haiku, GPT-4o)

**Safety features:**
- Step 1 tests all APIs with 1 call each (~$0.001) before running anything
- Results auto-save after each strategy — nothing is lost if it crashes
- Auto-retry on API errors (3 attempts with backoff)
- You can resume from where you left off

In [ ]:
!pip install -q openai anthropic google-genai matplotlib

In [ ]:
# ============================================================
# STEP 0: SET API KEYS
# ============================================================
import os

# Option A: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar > add your keys
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('Loaded keys from Colab Secrets')
except:
    print('Colab Secrets not available. Set keys manually below:')

# Option B: Set directly (uncomment and fill in)
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['GOOGLE_API_KEY'] = 'AI...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

for key in ['OPENAI_API_KEY', 'GOOGLE_API_KEY', 'ANTHROPIC_API_KEY']:
    val = os.environ.get(key, '')
    status = f'SET ({val[:10]}...)' if val else 'MISSING'
    print(f'  {key}: {status}')

In [ ]:
# ============================================================
# STEP 1: TEST ALL APIs (costs < $0.001)
# Run this FIRST. If any API fails, fix it before proceeding.
# ============================================================
import time

print('Testing each API with a tiny request...\n')
all_ok = True

# Test Google Gemini
try:
    from google import genai
    google_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    r = google_client.models.generate_content(
        model='gemini-2.5-flash', contents='Reply with just the word OK')
    print(f'  [T1] Gemini 2.5 Flash: PASS ("{r.text.strip()[:20]}")')
except Exception as e:
    print(f'  [T1] Gemini 2.5 Flash: FAIL - {e}')
    all_ok = False

# Test OpenAI GPT-4o-mini
try:
    from openai import OpenAI
    openai_client = OpenAI()
    r = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': 'Reply with just the word OK'}],
        max_tokens=5)
    print(f'  [T2] GPT-4o-mini: PASS ("{r.choices[0].message.content.strip()[:20]}")')
except Exception as e:
    print(f'  [T2] GPT-4o-mini: FAIL - {e}')
    all_ok = False

# Test Anthropic Claude Haiku
try:
    import anthropic
    anthropic_client = anthropic.Anthropic()
    r = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=5,
        messages=[{'role': 'user', 'content': 'Reply with just the word OK'}])
    print(f'  [T3] Claude Haiku: PASS ("{r.content[0].text.strip()[:20]}")')
except Exception as e:
    print(f'  [T3] Claude Haiku: FAIL - {e}')
    all_ok = False

# Test OpenAI GPT-4o
try:
    r = openai_client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': 'Reply with just the word OK'}],
        max_tokens=5)
    print(f'  [T4] GPT-4o: PASS ("{r.choices[0].message.content.strip()[:20]}")')
except Exception as e:
    print(f'  [T4] GPT-4o: FAIL - {e}')
    all_ok = False

print()
if all_ok:
    print('ALL APIs WORKING. Safe to proceed.')
else:
    print('SOME APIs FAILED. Fix them before running experiments.')
    print('Do NOT proceed to the next cell until all 4 pass.')

In [ ]:
# ============================================================
# STEP 2: LOAD ALL CODE
# ============================================================
import re
import json
import random
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

# ── Model Config ─────────────────────────────────────────
MODEL_TIERS = {
    'T1': {'name': 'Gemini 2.5 Flash', 'provider': 'google', 'model_id': 'gemini-2.5-flash',
           'input_cost_per_1m': 0.15, 'output_cost_per_1m': 0.60},
    'T2': {'name': 'GPT-4o-mini', 'provider': 'openai', 'model_id': 'gpt-4o-mini',
           'input_cost_per_1m': 0.15, 'output_cost_per_1m': 0.60},
    'T3': {'name': 'Claude Haiku', 'provider': 'anthropic', 'model_id': 'claude-haiku-4-5-20251001',
           'input_cost_per_1m': 0.80, 'output_cost_per_1m': 4.00},
    'T4': {'name': 'GPT-4o', 'provider': 'openai', 'model_id': 'gpt-4o',
           'input_cost_per_1m': 2.50, 'output_cost_per_1m': 10.00},
}

TYPE_DEFAULT_TIER = {'EXPL': 'T1', 'COMP': 'T2', 'LOC': 'T2', 'PATCH': 'T4', 'TEST': 'T3', 'VER': 'T1'}
SUBTASK_TYPES = ['EXPL', 'COMP', 'LOC', 'PATCH', 'TEST', 'VER']
MAX_TOKENS = 4096
TEMPERATURE = 0.0

DEFAULT_THRESHOLDS = {
    'T1': {'EXPL': 0.8, 'COMP': 0.4, 'LOC': 0.3, 'PATCH': 0.2, 'TEST': 0.3, 'VER': 0.7},
    'T2': {'EXPL': 0.9, 'COMP': 0.6, 'LOC': 0.5, 'PATCH': 0.3, 'TEST': 0.5, 'VER': 0.85},
    'T3': {'EXPL': 0.95, 'COMP': 0.8, 'LOC': 0.7, 'PATCH': 0.6, 'TEST': 0.7, 'VER': 0.95},
    'T4': {'EXPL': 1.0, 'COMP': 1.0, 'LOC': 1.0, 'PATCH': 1.0, 'TEST': 1.0, 'VER': 1.0},
}

# ── LLM Response ─────────────────────────────────────────
@dataclass
class LLMResponse:
    content: str
    input_tokens: int
    output_tokens: int
    cost: float
    model_id: str
    tier: str
    latency_ms: float
    success: bool = True
    error: Optional[str] = None

# ── API Callers with retry ────────────────────────────────
def compute_cost(tier, input_tokens, output_tokens):
    cfg = MODEL_TIERS[tier]
    return (input_tokens / 1_000_000) * cfg['input_cost_per_1m'] + \
           (output_tokens / 1_000_000) * cfg['output_cost_per_1m']

def _call_openai(model_id, system_prompt, user_prompt):
    from openai import OpenAI
    client = OpenAI()
    start = time.time()
    r = client.chat.completions.create(
        model=model_id, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user', 'content': user_prompt}])
    return {'content': r.choices[0].message.content or '',
            'input_tokens': r.usage.prompt_tokens,
            'output_tokens': r.usage.completion_tokens,
            'latency_ms': (time.time() - start) * 1000}

def _call_anthropic(model_id, system_prompt, user_prompt):
    import anthropic
    client = anthropic.Anthropic()
    start = time.time()
    r = client.messages.create(
        model=model_id, max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
        system=system_prompt,
        messages=[{'role': 'user', 'content': user_prompt}])
    return {'content': r.content[0].text if r.content else '',
            'input_tokens': r.usage.input_tokens,
            'output_tokens': r.usage.output_tokens,
            'latency_ms': (time.time() - start) * 1000}

def _call_google(model_id, system_prompt, user_prompt):
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    start = time.time()
    r = client.models.generate_content(
        model=model_id,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=TEMPERATURE,
            max_output_tokens=MAX_TOKENS,
        ),
    )
    latency = (time.time() - start) * 1000
    usage = r.usage_metadata
    return {'content': r.text if r.text else '',
            'input_tokens': usage.prompt_token_count,
            'output_tokens': usage.candidates_token_count,
            'latency_ms': latency}

_DISPATCH = {'openai': _call_openai, 'anthropic': _call_anthropic, 'google': _call_google}

def call_llm(tier, system_prompt, user_prompt, max_retries=3):
    """Call LLM with automatic retry on failure."""
    cfg = MODEL_TIERS[tier]
    fn = _DISPATCH[cfg['provider']]
    last_error = None
    for attempt in range(max_retries):
        try:
            result = fn(cfg['model_id'], system_prompt, user_prompt)
            cost = compute_cost(tier, result['input_tokens'], result['output_tokens'])
            return LLMResponse(
                content=result['content'], input_tokens=result['input_tokens'],
                output_tokens=result['output_tokens'], cost=cost,
                model_id=cfg['model_id'], tier=tier,
                latency_ms=result['latency_ms'], success=True)
        except Exception as e:
            last_error = str(e)
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f'      [retry {attempt+1}] {cfg["name"]}: {last_error[:60]}... waiting {wait}s')
                time.sleep(wait)
    return LLMResponse(
        content='', input_tokens=0, output_tokens=0, cost=0.0,
        model_id=cfg['model_id'], tier=tier, latency_ms=0.0,
        success=False, error=last_error)

# ── Sub-task prompts ──────────────────────────────────────
PROMPTS = {
    'EXPL': 'You are exploring a codebase.\n\nTask: {desc}\n\nCode:\n```python\n{code}\n```\n\nList the key components, functions, and their purposes.',
    'COMP': 'You are analyzing code.\n\nTask: {desc}\n\nCode:\n```python\n{code}\n```\n\nExplain what this code does and identify any bugs. Be specific.',
    'LOC':  'You are localizing a bug.\n\nTask: {desc}\n\nCode:\n```python\n{code}\n```\n\nIdentify the EXACT lines that are buggy and explain why.',
    'PATCH':'You are fixing code.\n\nTask: {desc}\n\nOriginal code:\n```python\n{code}\n```\n\nTests that must pass:\n```python\n{test}\n```\n\nWrite the COMPLETE fixed code. Output ONLY Python code in ```python``` fences.',
    'TEST': 'Write 3 additional edge-case tests.\n\nTask: {desc}\n\nCode:\n```python\n{code}\n```\n\nExisting tests:\n```python\n{test}\n```\n\nOutput ONLY Python test code in ```python``` fences.',
    'VER':  'Verify this fix.\n\nTask: {desc}\n\nOriginal:\n```python\n{orig}\n```\n\nFix:\n```python\n{fix}\n```\n\nTests:\n```python\n{test}\n```\n\nWill ALL tests pass? Answer: VERDICT: PASS or VERDICT: FAIL with explanation.',
}

print('All code loaded.')

In [ ]:
# ============================================================
# STEP 3: LOAD TASKS (20 coding tasks)
# ============================================================
TASKS = [
    {'id': 'bugfix_001', 'title': 'Fix off-by-one in pagination', 'difficulty': 0.2,
     'description': 'The paginate function returns one fewer item than expected.',
     'code': 'def paginate(items, page, per_page=10):\n    start = (page - 1) * per_page\n    end = start + per_page - 1\n    return items[start:end]\n\ndef total_pages(items, per_page=10):\n    return (len(items) + per_page - 1) // per_page',
     'test_code': 'def test_paginate():\n    items = list(range(25))\n    assert paginate(items, 1) == list(range(10))\n    assert paginate(items, 2) == list(range(10, 20))\n    assert paginate(items, 3) == list(range(20, 25))'},
    {'id': 'bugfix_002', 'title': 'Fix dict key error in user lookup', 'difficulty': 0.15,
     'description': 'get_user_email crashes when user has no email field.',
     'code': 'def get_user_email(users_db, user_id):\n    user = users_db.get(user_id)\n    return user["email"]',
     'test_code': 'def test_get_user_email():\n    db = {"u1": {"name": "Alice", "email": "a@b.com"}, "u2": {"name": "Bob"}}\n    assert get_user_email(db, "u1") == "a@b.com"\n    assert get_user_email(db, "u2") is None\n    assert get_user_email(db, "u3") is None'},
    {'id': 'bugfix_003', 'title': 'Fix race condition in counter', 'difficulty': 0.4,
     'description': 'SafeCounter is not actually thread-safe.',
     'code': 'import threading\n\nclass SafeCounter:\n    def __init__(self):\n        self.value = 0\n    def increment(self):\n        current = self.value\n        self.value = current + 1\n    def get(self):\n        return self.value',
     'test_code': 'import threading\ndef test_safe_counter():\n    counter = SafeCounter()\n    threads = []\n    for _ in range(100):\n        t = threading.Thread(target=lambda: [counter.increment() for _ in range(100)])\n        threads.append(t)\n    for t in threads: t.start()\n    for t in threads: t.join()\n    assert counter.get() == 10000'},
    {'id': 'bugfix_004', 'title': 'Fix timezone conversion', 'difficulty': 0.3,
     'description': 'to_utc adds offset instead of subtracting.',
     'code': 'from datetime import datetime, timedelta\ndef to_utc(dt_str, offset_hours):\n    dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S")\n    return dt + timedelta(hours=offset_hours)',
     'test_code': 'def test_to_utc():\n    result = to_utc("2024-03-10 14:00:00", -5)\n    assert result.hour == 19\n    result2 = to_utc("2024-03-10 10:00:00", 5)\n    assert result2.hour == 5'},
    {'id': 'bugfix_005', 'title': 'Fix binary search infinite loop', 'difficulty': 0.25,
     'description': 'binary_search has infinite loop due to missing +1/-1.',
     'code': 'def binary_search(arr, target):\n    low, high = 0, len(arr) - 1\n    while low <= high:\n        mid = (low + high) // 2\n        if arr[mid] < target: low = mid\n        elif arr[mid] > target: high = mid\n        else: return mid\n    return -1',
     'test_code': 'def test_binary_search():\n    arr = [1, 3, 5, 7, 9, 11, 13]\n    assert binary_search(arr, 7) == 3\n    assert binary_search(arr, 1) == 0\n    assert binary_search(arr, 13) == 6\n    assert binary_search(arr, 4) == -1'},
    {'id': 'bugfix_006', 'title': 'Fix merge sorted missing tail', 'difficulty': 0.15,
     'description': 'merge_sorted drops remaining elements.',
     'code': 'def merge_sorted(list1, list2):\n    result = []\n    i, j = 0, 0\n    while i < len(list1) and j < len(list2):\n        if list1[i] <= list2[j]:\n            result.append(list1[i]); i += 1\n        else:\n            result.append(list2[j]); j += 1\n    return result',
     'test_code': 'def test_merge_sorted():\n    assert merge_sorted([1,3,5], [2,4,6]) == [1,2,3,4,5,6]\n    assert merge_sorted([1,2], [3,4,5,6]) == [1,2,3,4,5,6]\n    assert merge_sorted([], [1,2]) == [1,2]\n    assert merge_sorted([1], []) == [1]'},
    {'id': 'bugfix_007', 'title': 'Fix cache with no eviction', 'difficulty': 0.5,
     'description': 'Cache class grows unbounded. Add LRU eviction.',
     'code': 'class Cache:\n    def __init__(self, max_size=100):\n        self.max_size = max_size\n        self.data = {}\n    def get(self, key): return self.data.get(key)\n    def put(self, key, value): self.data[key] = value',
     'test_code': 'def test_cache_eviction():\n    cache = Cache(max_size=3)\n    cache.put("a", 1); cache.put("b", 2); cache.put("c", 3)\n    cache.get("a")\n    cache.put("d", 4)\n    assert cache.get("a") == 1\n    assert cache.get("b") is None\n    assert cache.get("d") == 4'},
    {'id': 'bugfix_008', 'title': 'Fix CSV parsing with quotes', 'difficulty': 0.55,
     'description': 'parse_csv_line fails on commas inside quotes.',
     'code': 'def parse_csv_line(line):\n    return line.split(",")',
     'test_code': "def test_parse_csv():\n    assert parse_csv_line('a,b,c') == ['a', 'b', 'c']\n    assert parse_csv_line('\"hello, world\",b,c') == ['hello, world', 'b', 'c']\n    assert parse_csv_line('a,\"b,c\",d') == ['a', 'b,c', 'd']"},
    {'id': 'bugfix_009', 'title': 'Fix fibonacci stack overflow', 'difficulty': 0.25,
     'description': 'fibonacci has no memoization, overflows on large n.',
     'code': 'def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)',
     'test_code': 'def test_fibonacci():\n    assert fibonacci(0) == 0\n    assert fibonacci(1) == 1\n    assert fibonacci(10) == 55\n    assert fibonacci(50) == 12586269025'},
    {'id': 'bugfix_010', 'title': 'Fix SQL injection', 'difficulty': 0.45,
     'description': 'build_query uses string interpolation instead of parameterized queries.',
     'code': 'def build_query(table, filters):\n    query = f"SELECT * FROM {table}"\n    if filters:\n        conds = [f"{k} = \'{v}\'" for k,v in filters.items()]\n        query += " WHERE " + " AND ".join(conds)\n    return query',
     'test_code': 'def test_build_query_safe():\n    query, params = build_query("users", {"name": "Alice", "age": 30})\n    assert "?" in query or "%s" in query\n    assert "Alice" not in query\n    assert params == ["Alice", 30] or params == ("Alice", 30)'},
    {'id': 'feature_001', 'title': 'Implement retry decorator', 'difficulty': 0.4,
     'description': 'Create a retry decorator with exponential backoff.',
     'code': '# Implement: retry(max_retries=3, base_delay=1.0) decorator\n# Exponential backoff, re-raise last exception if all fail',
     'test_code': 'def test_retry():\n    count = 0\n    @retry(max_retries=3, base_delay=0.01)\n    def flaky():\n        nonlocal count; count += 1\n        if count < 3: raise ValueError("not yet")\n        return "ok"\n    assert flaky() == "ok"\n    assert count == 3'},
    {'id': 'feature_002', 'title': 'Implement rate limiter', 'difficulty': 0.55,
     'description': 'Create a token-bucket rate limiter.',
     'code': '# Implement: RateLimiter(rate=10, per=1.0)\n# try_acquire() returns True/False',
     'test_code': 'import time\ndef test_rate_limiter():\n    limiter = RateLimiter(rate=5, per=1.0)\n    for _ in range(5): assert limiter.try_acquire() is True\n    assert limiter.try_acquire() is False\ndef test_refill():\n    limiter = RateLimiter(rate=2, per=0.1)\n    limiter.try_acquire(); limiter.try_acquire()\n    assert limiter.try_acquire() is False\n    time.sleep(0.15)\n    assert limiter.try_acquire() is True'},
    {'id': 'feature_003', 'title': 'Implement trie autocomplete', 'difficulty': 0.5,
     'description': 'Create Trie with insert, search, starts_with, autocomplete.',
     'code': '# Implement: Trie class\n# insert(word), search(word)->bool, starts_with(prefix)->bool, autocomplete(prefix, limit=5)->list',
     'test_code': 'def test_trie():\n    trie = Trie()\n    for w in ["apple","app","application","bat","ball"]: trie.insert(w)\n    assert trie.search("app") is True\n    assert trie.search("ap") is False\n    assert trie.starts_with("app") is True\n    assert set(trie.autocomplete("app")) == {"app","apple","application"}'},
    {'id': 'feature_004', 'title': 'Implement event emitter', 'difficulty': 0.35,
     'description': 'Create EventEmitter with on, off, emit, once.',
     'code': '# Implement: EventEmitter class\n# on(event, cb), off(event, cb), emit(event, *args), once(event, cb)',
     'test_code': 'def test_emitter():\n    e = EventEmitter(); r = []\n    def cb(x): r.append(x)\n    e.on("d", cb); e.emit("d", 1); e.emit("d", 2)\n    assert r == [1, 2]\n    e.off("d", cb); e.emit("d", 3)\n    assert r == [1, 2]\ndef test_once():\n    e = EventEmitter(); r = []\n    e.once("p", lambda: r.append(1))\n    e.emit("p"); e.emit("p")\n    assert r == [1]'},
    {'id': 'feature_005', 'title': 'Implement JSON validator', 'difficulty': 0.5,
     'description': 'validate(data, schema) checks type, required fields, properties.',
     'code': '# Implement: validate(data, schema) -> (bool, list_of_errors)\n# Schema: {"type": "dict", "required": [...], "properties": {...}}',
     'test_code': 'def test_validate():\n    schema = {"type":"dict","required":["name","age"],"properties":{"name":{"type":"string"},"age":{"type":"int"}}}\n    ok,e = validate({"name":"Alice","age":30}, schema)\n    assert ok is True\n    ok2,e2 = validate({"name":"Bob"}, schema)\n    assert ok2 is False\n    ok3,e3 = validate({"name":123,"age":"x"}, schema)\n    assert ok3 is False and len(e3) >= 2'},
    {'id': 'refactor_001', 'title': 'Refactor nested conditionals', 'difficulty': 0.3,
     'description': 'Simplify deeply nested process_order using guard clauses.',
     'code': 'def process_order(order):\n    result = None\n    if order is not None:\n        if order.get("status") == "pending":\n            if order.get("items") and len(order["items"]) > 0:\n                if order.get("payment"):\n                    if order["payment"].get("verified"):\n                        total = sum(i["price"]*i["qty"] for i in order["items"])\n                        if total > 0: result = {"status":"processed","total":total}\n                        else: result = {"status":"error","message":"invalid total"}\n                    else: result = {"status":"error","message":"payment not verified"}\n                else: result = {"status":"error","message":"no payment info"}\n            else: result = {"status":"error","message":"no items"}\n        else: result = {"status":"error","message":"order not pending"}\n    else: result = {"status":"error","message":"no order"}\n    return result',
     'test_code': 'def test_process_order():\n    good = {"status":"pending","items":[{"price":10,"qty":2},{"price":5,"qty":1}],"payment":{"verified":True}}\n    r = process_order(good)\n    assert r["status"]=="processed" and r["total"]==25\n    assert process_order(None)["message"]=="no order"\n    assert process_order({"status":"shipped"})["message"]=="order not pending"'},
    {'id': 'complex_001', 'title': 'Implement URL shortener', 'difficulty': 0.6,
     'description': 'URLShortener with encode, decode, click tracking, stats.',
     'code': '# Implement: URLShortener class\n# encode(url)->6-char code, decode(code)->url, click(code), stats(code)->{url,clicks}',
     'test_code': 'def test_shortener():\n    s = URLShortener()\n    c = s.encode("https://example.com")\n    assert len(c) == 6\n    assert s.decode(c) == "https://example.com"\n    assert s.encode("https://example.com") == c\n    s.click(c); s.click(c)\n    assert s.stats(c)["clicks"] == 2\n    assert s.decode("xxxxxx") is None'},
    {'id': 'complex_002', 'title': 'Implement task scheduler', 'difficulty': 0.55,
     'description': 'TaskScheduler with topological sort and cycle detection.',
     'code': '# Implement: TaskScheduler\n# add_task(name, dependencies=[]), get_order()->list, detect cycles',
     'test_code': 'def test_scheduler():\n    s = TaskScheduler()\n    s.add_task("build",["compile"]); s.add_task("compile",["parse"])\n    s.add_task("parse",[]); s.add_task("test",["build"])\n    order = s.get_order()\n    assert order.index("parse") < order.index("compile")\n    assert order.index("compile") < order.index("build")\ndef test_cycle():\n    s = TaskScheduler()\n    s.add_task("a",["b"]); s.add_task("b",["a"])\n    try: s.get_order(); assert False\n    except ValueError: pass'},
    {'id': 'complex_003', 'title': 'Implement expression evaluator', 'difficulty': 0.7,
     'description': 'evaluate("2 + 3 * 4") with operator precedence and parens.',
     'code': '# Implement: evaluate(expr_string) -> number\n# Support +, -, *, /, parentheses, operator precedence',
     'test_code': 'def test_eval():\n    assert evaluate("2 + 3") == 5\n    assert evaluate("2 + 3 * 4") == 14\n    assert evaluate("(2 + 3) * 4") == 20\n    assert evaluate("10 / 2") == 5.0'},
    {'id': 'complex_004', 'title': 'Implement HTTP router', 'difficulty': 0.5,
     'description': 'Router with path params: /users/:id matches /users/42.',
     'code': '# Implement: Router class\n# add_route(method, path, handler), match(method, path)->(handler, params)',
     'test_code': 'def test_router():\n    r = Router()\n    r.add_route("GET","/users", lambda p: "all")\n    r.add_route("GET","/users/:id", lambda p: f"user {p[\' id\']}" )\n    h, p = r.match("GET","/users/42")\n    assert p == {"id": "42"}\n    h2, p2 = r.match("GET","/users")\n    assert h2(p2) == "all"\n    assert r.match("DELETE","/users") is None'},
]

print(f'Loaded {len(TASKS)} tasks')

In [ ]:
# ============================================================
# STEP 4: CORE FUNCTIONS
# ============================================================

def estimate_difficulty(task, subtask_type):
    code = task.get('code', '')
    test_code = task.get('test_code', '')
    code_lines = len(code.strip().split('\n')) if code.strip() else 0
    nf = len(re.findall(r'def \w+', code))
    nc = len(re.findall(r'class \w+', code))
    ni = 1 if re.search(r'^(import|from)', code, re.MULTILINE) else 0
    nd = max((len(l)-len(l.lstrip()))//4 for l in code.split('\n') if l.strip()) if code.strip() else 0
    ncond = len(re.findall(r'\b(if|elif|else|while|for)\b', code))
    na = len(re.findall(r'assert', test_code))
    base = task.get('difficulty', 0.5)
    mult = {'EXPL':0.3,'COMP':0.5,'LOC':0.6,'PATCH':1.0,'TEST':0.7,'VER':0.4}
    raw = 0.3*base + 0.1*min(code_lines/50,1) + 0.1*min(nf/5,1) + 0.1*min(nd/5,1) + \
          0.1*min(ncond/8,1) + 0.1*min(na/6,1) + 0.1*ni + 0.1*min(nc/2,1)
    return round(min(raw * mult.get(subtask_type, 1.0), 1.0), 3)

def select_tier(subtask_type, difficulty):
    for tier in ['T1','T2','T3','T4']:
        if DEFAULT_THRESHOLDS[tier][subtask_type] >= difficulty:
            return tier
    return 'T4'

def confidence_check(st, resp, task):
    if not resp.success or not resp.content.strip(): return False
    c = resp.content.strip()
    checks = {
        'EXPL': len(c) > 20,
        'COMP': len(c) > 50,
        'LOC': any(w in c.lower() for w in ['line','bug','issue','error']),
        'PATCH': ('```python' in c or 'def ' in c or 'class ' in c) and len(c) > 30,
        'TEST': 'def test_' in c or 'assert' in c,
        'VER': any(w in c.upper() for w in ['VERDICT','PASS','FAIL']),
    }
    return checks.get(st, True)

def extract_code(text):
    m = re.findall(r'```python\s*\n(.*?)```', text, re.DOTALL)
    if m: return m[0].strip()
    m = re.findall(r'```\s*\n(.*?)```', text, re.DOTALL)
    if m: return m[0].strip()
    lines = []; started = False
    for l in text.split('\n'):
        if l.strip().startswith(('def ','class ','import ','from ')): started = True
        if started: lines.append(l)
    return '\n'.join(lines).strip() if lines else text.strip()

def evaluate_patch(task, patch_code):
    if not patch_code.strip(): return False
    test_code = task.get('test_code', '')
    if not test_code.strip(): return False
    full = patch_code + '\n\n' + test_code
    funcs = re.findall(r'def (test_\w+)', test_code)
    ns = {}
    try:
        exec(full, ns)
        for f in funcs:
            if f in ns: ns[f]()
        return True
    except: return False

def run_subtask(task, st, tier, extra=None):
    extra = extra or {}
    sys_prompt = 'You are an expert software engineer. Be precise. Follow instructions exactly.'
    if st == 'VER':
        user_prompt = PROMPTS[st].format(desc=task['description'], orig=extra.get('original_code',task['code']),
                                         fix=extra.get('fixed_code',''), test=task['test_code'])
    else:
        user_prompt = PROMPTS[st].format(desc=task['description'], code=task['code'], test=task.get('test_code',''))
    return call_llm(tier, sys_prompt, user_prompt)

print('Core functions loaded.')

In [ ]:
# ============================================================
# STEP 5: RUN EXPERIMENTS (auto-saves after each strategy)
# ============================================================
STRATEGIES = [
    {'name': 'single_frontier', 'strategy': 'single', 'tier': 'T4'},
    {'name': 'single_small',    'strategy': 'single', 'tier': 'T1'},
    {'name': 'single_medium',   'strategy': 'single', 'tier': 'T2'},
    {'name': 'single_large',    'strategy': 'single', 'tier': 'T3'},
    {'name': 'random_routing',  'strategy': 'random', 'tier': None},
    {'name': 'type_only',       'strategy': 'type_only', 'tier': None},
    {'name': 'taskrouter',      'strategy': 'taskrouter', 'tier': None},
]

# Load previous results if resuming
try:
    with open('all_results.json') as f:
        all_results = json.load(f)
    print(f'Loaded previous results: {list(all_results.keys())}')
except:
    all_results = {}
    print('Starting fresh.')

running_cost = 0.0

for strat in STRATEGIES:
    name = strat['name']

    # Skip if already done
    if name in all_results:
        print(f'\n--- {name}: ALREADY DONE (skipping) ---')
        continue

    print(f'\n{"="*60}')
    print(f'Strategy: {name} | Running cost so far: ${running_cost:.4f}')
    print(f'{"="*60}')

    task_results = []
    strat_cost = 0
    resolved = 0
    per_type = {t: {'count':0,'to_T1':0,'to_T2':0,'to_T3':0,'to_T4':0,'cascaded':0} for t in SUBTASK_TYPES}

    for i, task in enumerate(TASKS):
        print(f'  [{i+1}/{len(TASKS)}] {task["id"]}: {task["title"]}', end=' ')
        result = {'task_id': task['id'], 'decisions': [], 'patch_code': '',
                  'test_passed': False, 'total_cost': 0, 'total_input_tokens': 0,
                  'total_output_tokens': 0, 'wasted_cost': 0}
        extra_ctx = {}

        for st in SUBTASK_TYPES:
            # Select tier
            if strat['strategy'] == 'single': init_tier = strat['tier']
            elif strat['strategy'] == 'random': init_tier = random.choice(['T1','T2','T3','T4'])
            elif strat['strategy'] == 'type_only': init_tier = TYPE_DEFAULT_TIER[st]
            else:
                d = estimate_difficulty(task, st)
                init_tier = select_tier(st, d)

            # Execute with cascading
            cur = init_tier
            cascades = 0
            w_tok = 0; w_cost = 0
            tiers = ['T1','T2','T3','T4']
            si = tiers.index(cur)
            final = None

            for ti in range(si, 4):
                cur = tiers[ti]
                resp = run_subtask(task, st, cur, extra_ctx)
                if strat['strategy'] in ('single','random') or confidence_check(st, resp, task):
                    final = resp; break
                else:
                    w_tok += resp.input_tokens + resp.output_tokens
                    w_cost += resp.cost
                    cascades += 1
            if not final: final = resp

            dec = {'subtask_type': st, 'initial_tier': init_tier, 'final_tier': cur,
                   'cascaded': cascades > 0, 'cascade_depth': cascades,
                   'model_id': final.model_id, 'input_tokens': final.input_tokens,
                   'output_tokens': final.output_tokens, 'cost': round(final.cost,6),
                   'latency_ms': round(final.latency_ms,1),
                   'wasted_tokens': w_tok, 'wasted_cost': round(w_cost,6)}
            result['decisions'].append(dec)
            result['total_cost'] += final.cost + w_cost
            result['total_input_tokens'] += final.input_tokens + w_tok
            result['total_output_tokens'] += final.output_tokens
            result['wasted_cost'] += w_cost

            if st == 'PATCH':
                result['patch_code'] = extract_code(final.content)
                extra_ctx['fixed_code'] = result['patch_code']
                extra_ctx['original_code'] = task['code']

            per_type[st]['count'] += 1
            per_type[st][f'to_{cur}'] += 1
            if cascades > 0: per_type[st]['cascaded'] += 1

        result['test_passed'] = evaluate_patch(task, result['patch_code'])
        result['total_cost'] = round(result['total_cost'], 6)
        result['wasted_cost'] = round(result['wasted_cost'], 6)
        task_results.append(result)
        strat_cost += result['total_cost']
        if result['test_passed']: resolved += 1
        print(f'| {"PASS" if result["test_passed"] else "FAIL"} | ${result["total_cost"]:.4f}')

    # Compute summary
    rr = round(resolved/len(TASKS)*100, 1)
    ts = sum(s['count'] for s in per_type.values())
    tc = sum(s['cascaded'] for s in per_type.values())
    tw = round(sum(r['wasted_cost'] for r in task_results), 4)
    pt = {}
    for st, s in per_type.items():
        c = s['count']
        if c == 0: continue
        pt[st] = {k: round(s[k]/c*100,1) if k != 'count' else c
                  for k in ['count','to_T1','to_T2','to_T3','to_T4','cascaded']}
        for k in ['to_T1','to_T2','to_T3','to_T4','cascaded']:
            pt[st][f'pct_{k.replace("to_","")}' if 'to_' in k else 'cascade_pct'] = round(s[k]/c*100,1)

    all_results[name] = {
        'strategy': name, 'num_tasks': len(TASKS), 'task_results': task_results,
        'summary': {
            'resolve_rate': rr, 'total_resolved': resolved,
            'total_cost': round(strat_cost, 4),
            'cost_per_resolved': round(strat_cost/resolved, 4) if resolved > 0 else 999,
            'total_input_tokens': sum(r['total_input_tokens'] for r in task_results),
            'total_output_tokens': sum(r['total_output_tokens'] for r in task_results),
            'total_wasted_cost': tw, 'cascade_rate': round(tc/ts*100,1) if ts>0 else 0,
            'total_subtasks': ts, 'total_cascades': tc,
            'per_type': {st: {'count': per_type[st]['count'],
                              'pct_T1': round(per_type[st]['to_T1']/per_type[st]['count']*100,1) if per_type[st]['count']>0 else 0,
                              'pct_T2': round(per_type[st]['to_T2']/per_type[st]['count']*100,1) if per_type[st]['count']>0 else 0,
                              'pct_T3': round(per_type[st]['to_T3']/per_type[st]['count']*100,1) if per_type[st]['count']>0 else 0,
                              'pct_T4': round(per_type[st]['to_T4']/per_type[st]['count']*100,1) if per_type[st]['count']>0 else 0,
                              'cascade_pct': round(per_type[st]['cascaded']/per_type[st]['count']*100,1) if per_type[st]['count']>0 else 0}
                        for st in SUBTASK_TYPES if per_type[st]['count'] > 0},
        },
    }
    running_cost += strat_cost

    # AUTO-SAVE after each strategy
    with open('all_results.json', 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f'\n  {name}: {rr}% resolved | ${strat_cost:.4f} cost | SAVED')

print(f'\n{"="*60}')
print(f'ALL DONE. Total cost: ${running_cost:.4f}')
print(f'{"="*60}')

In [ ]:
# ============================================================
# STEP 6: RESULTS SUMMARY
# ============================================================
baseline = all_results['single_frontier']['summary']['total_cost']

print(f'{"Strategy":<25} {"Resolve%":>10} {"Cost($)":>10} {"$/Resolved":>12} {"Savings":>10}')
print('=' * 70)
for name in ['single_frontier','single_small','single_medium','single_large',
             'random_routing','type_only','taskrouter']:
    if name not in all_results: continue
    s = all_results[name]['summary']
    red = ((baseline - s['total_cost'])/baseline*100) if baseline > 0 else 0
    r = '---' if name == 'single_frontier' else f'{red:.1f}%'
    cpr = f'${s["cost_per_resolved"]:.4f}' if s['cost_per_resolved'] < 100 else 'N/A'
    print(f'{name:<25} {s["resolve_rate"]:>9.1f}% {s["total_cost"]:>9.4f} {cpr:>12} {r:>10}')

In [ ]:
# ============================================================
# STEP 7: PER-TYPE ROUTING TABLE
# ============================================================
if 'taskrouter' in all_results:
    print(f'{"Type":<8} {"% T1":>8} {"% T2":>8} {"% T3":>8} {"% T4":>8} {"Cascade%":>10}')
    print('-' * 55)
    for st in SUBTASK_TYPES:
        p = all_results['taskrouter']['summary']['per_type'].get(st, {})
        if p:
            print(f'{st:<8} {p["pct_T1"]:>7.1f}% {p["pct_T2"]:>7.1f}% {p["pct_T3"]:>7.1f}% {p["pct_T4"]:>7.1f}% {p["cascade_pct"]:>9.1f}%')

In [ ]:
# ============================================================
# STEP 8: FIGURES
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'font.family': 'serif'})

dn = {'single_frontier':'Frontier (T4)','single_small':'Small (T1)','single_medium':'Medium (T2)',
      'single_large':'Large (T3)','random_routing':'Random','type_only':'Type-Only','taskrouter':'TaskRouter'}

# Pareto frontier
fig, ax = plt.subplots(figsize=(8,5))
for name, data in all_results.items():
    s = data['summary']
    is_tr = name == 'taskrouter'
    ax.scatter(s['total_cost'], s['resolve_rate'], s=200 if is_tr else 80,
               marker='*' if is_tr else 'o', color='#e74c3c' if is_tr else '#3498db',
               zorder=5, edgecolors='black', linewidths=0.5)
    ax.annotate(dn.get(name,name), (s['total_cost'], s['resolve_rate']),
                textcoords='offset points', xytext=(8,5), fontsize=9)
ax.set_xlabel('Total Cost ($)'); ax.set_ylabel('Resolve Rate (%)')
ax.set_title('Cost-Quality Tradeoff'); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('pareto_frontier.png', dpi=300); plt.savefig('pareto_frontier.pdf')
print('Saved pareto_frontier.png + pdf'); plt.show()

# Token distribution
if 'taskrouter' in all_results:
    tt = {}
    for tr in all_results['taskrouter']['task_results']:
        for d in tr['decisions']:
            st = d['subtask_type']; tt[st] = tt.get(st,0) + d['input_tokens'] + d['output_tokens']
    fig, ax = plt.subplots(figsize=(7,5))
    ax.pie(list(tt.values()), labels=list(tt.keys()),
           colors=['#3498db','#2ecc71','#e74c3c','#f39c12','#9b59b6','#1abc9c'],
           autopct='%1.1f%%', startangle=90)
    ax.set_title('Token Distribution by Sub-Task Type')
    plt.tight_layout(); plt.savefig('token_dist.png', dpi=300)
    print('Saved token_dist.png'); plt.show()

# Bar chart comparison
order = ['single_frontier','single_large','single_medium','single_small','random_routing','type_only','taskrouter']
labels = [dn.get(n,n) for n in order if n in all_results]
costs = [all_results[n]['summary']['total_cost'] for n in order if n in all_results]
rates = [all_results[n]['summary']['resolve_rate'] for n in order if n in all_results]
colors = ['#e74c3c' if 'TaskRouter' in l else '#3498db' for l in labels]
fig, (a1,a2) = plt.subplots(1,2,figsize=(14,5))
a1.barh(labels, costs, color=colors); a1.set_xlabel('Total Cost ($)'); a1.set_title('Cost'); a1.invert_yaxis()
a2.barh(labels, rates, color=colors); a2.set_xlabel('Resolve Rate (%)'); a2.set_title('Resolve Rate'); a2.invert_yaxis()
plt.tight_layout(); plt.savefig('comparison.png', dpi=300)
print('Saved comparison.png'); plt.show()

In [ ]:
# ============================================================
# STEP 9: GENERATE LATEX TABLES (copy into main.tex)
# ============================================================
print('='*60)
print('LATEX TABLES - Copy these into main.tex')
print('='*60)

# Table: Overall Performance
print('\n% === TABLE: Overall Performance ===')
header = r'\begin{table}[htbp]' + '\n' + r'\caption{Overall Performance Comparison (%d tasks)}' % len(TASKS)
header += '\n' + r'\label{tab:overall}' + '\n' + r'\centering' + '\n'
header += r'\begin{tabular}{lrrrr}' + '\n' + r'\toprule'
header += '\n' + r'\textbf{Method} & \textbf{Resolve \%} & \textbf{Cost (\$)} & \textbf{\$/Resolved} & \textbf{Reduction} \\'
header += '\n' + r'\midrule'
print(header)

ln = {'single_frontier':'Single-Frontier (T4)','single_small':'Single-Small (T1)',
      'single_medium':'Single-Medium (T2)','single_large':'Single-Large (T3)',
      'random_routing':'Random Routing','type_only':'Type-Only Routing',
      'taskrouter':r'\textbf{TaskRouter}'}

for n in ['single_frontier','single_small','single_medium','single_large',
          'random_routing','type_only','taskrouter']:
    if n not in all_results: continue
    s = all_results[n]['summary']
    red = ((baseline-s['total_cost'])/baseline*100) if baseline>0 else 0
    rs = '---' if n=='single_frontier' else f'{red:.1f}\\%'
    cpr = f'\\${s["cost_per_resolved"]:.4f}' if s['cost_per_resolved']<100 else 'N/A'
    if n == 'type_only': print(r'\midrule')
    print(f'{ln[n]} & {s["resolve_rate"]:.1f}\\% & \\${s["total_cost"]:.4f} & {cpr} & {rs} \\\\')

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

# Table: Per-Type Routing
print('\n% === TABLE: Per-Type Routing ===')
print(r'\begin{table}[htbp]')
print(r'\caption{Per-Sub-Task Routing Behavior (TaskRouter)}')
print(r'\label{tab:pertask}')
print(r'\centering')
print(r'\begin{tabular}{lrrrrr}')
print(r'\toprule')
print(r'\textbf{Type} & \textbf{\% T1} & \textbf{\% T2} & \textbf{\% T3} & \textbf{\% T4} & \textbf{Cascade \%} \\')
print(r'\midrule')
for st in SUBTASK_TYPES:
    p = all_results.get('taskrouter',{}).get('summary',{}).get('per_type',{}).get(st,{})
    if p:
        print(f'{st} & {p["pct_T1"]:.1f}\\% & {p["pct_T2"]:.1f}\\% & {p["pct_T3"]:.1f}\\% & {p["pct_T4"]:.1f}\\% & {p["cascade_pct"]:.1f}\\% \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

# Cascading stats
if 'taskrouter' in all_results:
    ts = all_results['taskrouter']['summary']
    net = baseline - ts['total_cost']
    print(f'\n% === Cascading Stats ===')
    print(f'% Total sub-tasks: {ts["total_subtasks"]}')
    print(f'% Cascades: {ts["total_cascades"]} ({ts["cascade_rate"]}%)')
    print(f'% Wasted cost: ${ts["total_wasted_cost"]}')
    print(f'% Net savings vs frontier: ${net:.4f}')

print('\n% === Abstract placeholder values ===')
if 'taskrouter' in all_results:
    tr = all_results['taskrouter']['summary']
    sf = all_results['single_frontier']['summary']
    red = ((sf['total_cost']-tr['total_cost'])/sf['total_cost']*100)
    retain = (tr['resolve_rate']/sf['resolve_rate']*100) if sf['resolve_rate']>0 else 0
    overhead = (tr['total_wasted_cost']/tr['total_cost']*100) if tr['total_cost']>0 else 0
    print(f'% X% cost reduction = {red:.1f}%')
    print(f'% Y% of baseline resolve rate = {retain:.1f}%')
    print(f'% Z% routing overhead = {overhead:.1f}%')

In [ ]:
# ============================================================
# STEP 10: DOWNLOAD FILES
# ============================================================
try:
    from google.colab import files
    files.download('all_results.json')
    files.download('pareto_frontier.png')
    files.download('pareto_frontier.pdf')
    files.download('token_dist.png')
    files.download('comparison.png')
    print('Files downloaded!')
except:
    print('Files saved locally (not in Colab).')

print('\n=== NEXT STEPS ===')
print('1. Copy the LaTeX tables from Step 9 into main.tex')
print('2. Replace the abstract [X%], [Y%], [Z%] placeholders')
print('3. Upload main.tex + references.bib + figures to Overleaf')
print('4. Compile and review your paper!')